# Kaggle - Brain Tumor MRI Dataset

You can find the dataset and some informations about on the [Kaggle page](https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset).

For details on steps below, please see documentation in the *docs* directory.

## Kaggle notebook setup
### Installations

In [1]:
!pip install mlflow --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 88.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 99.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 69.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 796.7/796.7 kB 51.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 5.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour

### Data location

In [2]:
import os
print(os.listdir('/kaggle/input/'))
print(os.listdir('/kaggle/input/radimagenet-densenet121-notop'))
print(os.listdir('/kaggle/input/brain-tumor-mri-preprocessed'))
print(os.listdir('/kaggle/input/brain-tumor-mri-preprocessed/processed'))

['brain-tumor-mri-preprocessed', 'radimagenet-densenet121-notop']
['RadImageNet-DenseNet121_notop.h5']
['processed']
['Validation', 'Training', 'Testing', '.gitkeep']


## General

In [3]:
import mlflow
from mlflow.tracking import MlflowClient
mlflow.set_tracking_uri("https://preachingly-nonabjuratory-marget.ngrok-free.dev")
mlflow.set_experiment("Brain_Tumor_Training")

<Experiment: artifact_location='mlflow-artifacts:/830660600119881173', creation_time=1769099937797, experiment_id='830660600119881173', last_update_time=1769099937797, lifecycle_stage='active', name='Brain_Tumor_Training', tags={'mlflow.experimentKind': 'custom_model_development'}>

In [4]:
import numpy as np

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping

import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from datetime import datetime
import math

2026-02-03 15:23:30.985310: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770132211.187171      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770132211.239620      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770132211.693504      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770132211.693540      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770132211.693543      55 computation_placer.cc:177] computation placer alr

In [5]:
tf.keras.mixed_precision.set_global_policy("mixed_float16")
print("Num GPUs Available:", len(tf.config.list_physical_devices('GPU')))
tf.config.list_physical_devices('GPU')

Num GPUs Available: 1


[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

In [6]:
# path management
PROJECT_ROOT = '/kaggle/input'
PREP_DIR = PROJECT_ROOT + '/brain-tumor-mri-preprocessed/processed'
ARTEFACTS_DIR = PROJECT_ROOT + '/radimagenet-densenet121-notop'

CLASSES = ["notumor", "glioma", "meningioma", "pituitary"]

# parameters
IMG_SIZE = 260
SEED = 42

PROJECT_NAME = "BrainTumorMRI"
MODEL_TYPE = "DenseNet121"
TWO_HEAD = True
MODEL_NAME = f"{PROJECT_NAME}_{MODEL_TYPE}_{TWO_HEAD*"2Head"}"
FREEZE_BACKBONE = True
MASK_TUMOR_TYPE_LOSS = True
BATCH_SIZE = 32
DATA_AUGMENTATION = True

## Modeling

### Backbone

In [7]:
# 1. Create DenseNet121 WITHOUT weights
backbone = DenseNet121(
    include_top=False,
    weights=None,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

# 2. Load RadImageNet weights
backbone.load_weights(ARTEFACTS_DIR + "/RadImageNet-DenseNet121_notop.h5")

# 3. Freeze the backbone for firsts training
backbone.trainable = not FREEZE_BACKBONE

print("✅ RadImageNet DenseNet121 loaded successfully")

I0000 00:00:1770132223.464434      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


✅ RadImageNet DenseNet121 loaded successfully


In [8]:
#backbone.summary()

In [9]:
w = backbone.weights[0].numpy()
print("Mean:", np.mean(w), "Std:", np.std(w))

Mean: -0.0028284893 Std: 0.10494729


Model seems to be correctly loaded.

### Model definition

In [10]:
model_data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal", seed=SEED),
    layers.RandomZoom((-0.03,0.03),(-0.03,0.03), seed=SEED),
    layers.RandomTranslation((-0.01,0.01),(-0.01,0.01), seed=SEED),
], name='data_augmentation_part')

In [11]:
model_head1 = keras.Sequential([
    layers.Dense(128, use_bias=False), # TO TEST : 256 ?
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.Dropout(0.2),
    layers.Dense(1,activation='sigmoid')
], name='tumor_presence')

In [12]:
model_head2 = keras.Sequential([
    layers.Dense(128, use_bias=False), # TO TEST : 256 ?
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.Dropout(0.2),
    layers.Dense(4,activation='softmax')
], name='tumor_type')

In [13]:
def shared_head_part(inputs, backbone, data_augmentation):
    # Data augmentation (training only)
    x = data_augmentation(inputs)
    # Backbone - force into inference
    x = backbone(x, training=False)

    # Shared head
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(512, use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(0.4)(x)

    return x

In [14]:
inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))

x = shared_head_part(inputs, backbone, model_data_augmentation)

#Heads
output_presence = model_head1(x)
output_type = model_head2(x)

model = keras.Model(
    inputs=inputs,
    outputs={
        "tumor_presence": output_presence,
        "tumor_type": output_type
    },
    name='densenet_two_head'
)

In [15]:
loss_presence = keras.losses.BinaryFocalCrossentropy(
    gamma=2.0,
    alpha=0.25 # to favorize tumor detection (penalize false negatives), but taking account that tumors are 75% of data
)

In [16]:
def masked_sparse_cce(y_true, y_pred):
    tumor_present = tf.cast(y_true != 0, tf.float32)
    loss = tf.keras.losses.sparse_categorical_crossentropy(y_true, y_pred)
    loss = loss * tumor_present
    return tf.reduce_sum(loss) / (tf.reduce_sum(tumor_present) + 1e-6)

In [17]:
loss_weight_presence = 1.0
loss_weight_type = 1.3 # we give a little more weight to the classification of the type

model.compile(
    optimizer=keras.optimizers.Adam(), # change learning_rate for 1e-4 in fine-tuning steps
    loss={
        "tumor_presence": loss_presence,
        "tumor_type": masked_sparse_cce,
    },
    
    loss_weights={
        "tumor_presence": loss_weight_presence,
        "tumor_type": loss_weight_type, 
    },
    
    metrics={
        "tumor_presence": [
            keras.metrics.BinaryAccuracy(name="accuracy"),
            keras.metrics.Recall(name="recall"),
            keras.metrics.Precision(name="precision"),
            #keras.metrics.F1Score(name="f1_score"),
            keras.metrics.AUC(name="auc")
        ],
        "tumor_type": [
            "accuracy", 
            #"f1_score"
        ],
    }
)

In [18]:
#model.summary()

## Streaming Training

In [19]:
def parse_tfrecord(example_proto):
    """
    Parse a single TFRecord example and convert grayscale → RGB.
    """
    feature_description = {
        "image": tf.io.FixedLenFeature([], tf.string),
        "label": tf.io.FixedLenFeature([], tf.int64),
    }

    example = tf.io.parse_single_example(example_proto, feature_description)

    # Deserialize image
    image = tf.io.parse_tensor(example["image"], out_type=tf.float32)

    # Shape after loading: (260, 260, 3)
    image.set_shape((260, 260, 3))

    label = tf.cast(example["label"], tf.int32)

    return image, label



def load_tfrecord_dataset(tfrecord_dir, shuffle=False, batch_size=1, repeat=False):
    """
    Load a TFRecord dataset from a directory.

    Args:
        tfrecord_dir (str or Path): Folder containing .tfrecord files
        shuffle (bool): Whether to shuffle files and samples
        batch_size (int): Batch size (can stay 1)
        repeat (bool): Repeat dataset indefinitely (for training)

    Returns:
        tf.data.Dataset
    """
    tfrecord_files = tf.io.gfile.glob(
        str(tfrecord_dir) + "/*.tfrecord"
    )

    ds = tf.data.TFRecordDataset(
        tfrecord_files,
        num_parallel_reads=tf.data.AUTOTUNE
    )

    ds = ds.map(
        parse_tfrecord,
        num_parallel_calls=tf.data.AUTOTUNE
    )

    if shuffle:
        ds = ds.shuffle(buffer_size=512)

    if repeat:
        ds = ds.repeat()

    ds = ds.batch(batch_size)
    ds = ds.prefetch(tf.data.AUTOTUNE)

    return ds


In [20]:
TRAIN_DIR = PREP_DIR + "/Training"
VAL_DIR   = PREP_DIR + "/Validation"
TEST_DIR  = PREP_DIR + "/Testing"

train_ds = load_tfrecord_dataset(
    TRAIN_DIR,
    shuffle=True,
    batch_size=BATCH_SIZE,
    repeat=False
).prefetch(tf.data.AUTOTUNE)

val_ds = load_tfrecord_dataset(
    VAL_DIR,
    shuffle=False,
    batch_size=BATCH_SIZE,
    repeat=False
).prefetch(tf.data.AUTOTUNE)

"""
test_ds = load_tfrecord_dataset(
    TEST_DIR,
    shuffle=False,
    batch_size=BATCH_SIZE,
    repeat=False
).prefetch(tf.data.AUTOTUNE)
"""

'\ntest_ds = load_tfrecord_dataset(\n    TEST_DIR,\n    shuffle=False,\n    batch_size=BATCH_SIZE,\n    repeat=False\n).prefetch(tf.data.AUTOTUNE)\n'

In [21]:
def split_labels(image, label):
    """
    Create labels for a 2-head model.
    """
    tumor_present = tf.cast(label != 0, tf.float32)
    tumor_type = tf.cast(label, tf.int32)

    return image, {
        "tumor_presence": tumor_present,
        "tumor_type": tumor_type
    }


In [22]:
train_ds = train_ds.map(split_labels, num_parallel_calls=tf.data.AUTOTUNE)
val_ds = val_ds.map(split_labels, num_parallel_calls=tf.data.AUTOTUNE)

In [23]:
#for x, y in train_ds.take(1):
#    print("Image:")
#    print(x.dtype, x.shape)
#    print("\nLabels:")
#    for k, v in y.items():
#        print(k, v.dtype, v.shape)

In [24]:
def count_tfrecord_batches(directory_path, batch_size):
    """
    Count number of batches for a TFRecord dataset.

    Assumes:
    - 1 sample per TFRecord
    - batch_size is variable

    Returns:
    - number of batches = ceil(num_samples / batch_size)
    """
    directory_path = Path(directory_path)
    n_samples = len(list(directory_path.glob("*.tfrecord")))
    return math.ceil(n_samples / batch_size)


In [25]:
print(train_ds.element_spec)

(TensorSpec(shape=(None, 260, 260, 3), dtype=tf.float32, name=None), {'tumor_presence': TensorSpec(shape=(None,), dtype=tf.float32, name=None), 'tumor_type': TensorSpec(shape=(None,), dtype=tf.int32, name=None)})


In [26]:
#to_monitor = "val_tumor_presence_recall"
to_monitor = "val_tumor_type_loss"

reduce_lr = ReduceLROnPlateau(
    monitor=to_monitor,
    mode="max",
    factor=0.5,
    patience=5,
    min_lr=1e-6,
    verbose=1
)

early_stopping = EarlyStopping(
    monitor=to_monitor,
    mode="max",
    min_delta=0.0001,
    patience=10,
    restore_best_weights=True,
    verbose=1,
)

In [27]:
#print("RUN_NAME:", RUN_NAME)
#print("TRAIN steps:", count_tfrecord_batches(TRAIN_DIR, BATCH_SIZE))
#print("VAL steps:", count_tfrecord_batches(VAL_DIR, BATCH_SIZE))
#print("train_ds element_spec:", train_ds.element_spec)

In [28]:
RUN_NAME = (
    f"{MODEL_TYPE}"
    f"freeze={FREEZE_BACKBONE}_"
    f"mask={MASK_TUMOR_TYPE_LOSS}_"
    f"{datetime.now().strftime('%Y%m%d-%H%M')}"
)

print(f"Run name: {RUN_NAME}\n")

with mlflow.start_run(run_name=RUN_NAME):

    mlflow.tensorflow.autolog(registered_model_name=MODEL_NAME)

    mlflow.log_params({
        "project": PROJECT_NAME,
        "model_type": MODEL_TYPE,
        "pretrained_weights": "RadImageNet",
        "backbone_frozen": FREEZE_BACKBONE,
        "head_1": "tumor_presence_binary",
        "head_2": "tumor_type_softmax",
        "loss_weight_presence": loss_weight_presence,
        "loss_weight_type": loss_weight_type,
        "mask_tumor_type_loss": MASK_TUMOR_TYPE_LOSS,
        "label_0_means": "no_tumor",
        "data_augmentation": DATA_AUGMENTATION,
    })

    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=50,
        #steps_per_epoch=count_tfrecord_batches(TRAIN_DIR, BATCH_SIZE),
        #validation_steps=count_tfrecord_batches(VAL_DIR, BATCH_SIZE),
        callbacks=[reduce_lr, early_stopping],
        verbose=1,
    )

    client = MlflowClient()
    latest_version = client.get_latest_versions(MODEL_NAME)[-1].version

    client.transition_model_version_stage(
        name=MODEL_NAME,
        version=latest_version,
        stage="Staging"
    )


Run name: DenseNet121freeze=True_mask=True_20260203-1523



2026/02/03 15:24:07 WARNING mlflow.data.tensorflow_dataset: Failed to infer schema for TensorFlow dataset. Exception: Failed to infer schema for tf.data.Dataset. Schemas can only be inferred if the dataset consists of tensors. Ragged tensors, tensor arrays, and other types are not supported. Additionally, datasets with nested tensors are not supported.
2026/02/03 15:24:10 WARNING mlflow.data.tensorflow_dataset: Failed to infer schema for TensorFlow dataset. Exception: Failed to infer schema for tf.data.Dataset. Schemas can only be inferred if the dataset consists of tensors. Ragged tensors, tensor arrays, and other types are not supported. Additionally, datasets with nested tensors are not supported.


Epoch 1/50


I0000 00:00:1770132269.946050     138 cuda_dnn.cc:529] Loaded cuDNN version 91002


    143/Unknown 46s 158ms/step - loss: 1.5082 - tumor_presence_accuracy: 0.7618 - tumor_presence_auc: 0.7999 - tumor_presence_loss: 0.1780 - tumor_presence_precision: 0.8596 - tumor_presence_recall: 0.7918 - tumor_type_accuracy: 0.4705 - tumor_type_loss: 1.0232

...

143/143 ━━━━━━━━━━━━━━━━━━━━ 47s 325ms/step - loss: 0.5145 - tumor_presence_accuracy: 0.9465 - tumor_presence_auc: 0.9818 - tumor_presence_loss: 0.0406 - tumor_presence_precision: 0.9604 - tumor_presence_recall: 0.9667 - tumor_type_accuracy: 0.6194 - tumor_type_loss: 0.3645 - val_loss: 0.6727 - val_tumor_presence_accuracy: 0.9414 - val_tumor_presence_auc: 0.9797 - val_tumor_presence_loss: 0.0448 - val_tumor_presence_precision: 0.9316 - val_tumor_presence_recall: 0.9915 - val_tumor_type_accuracy: 0.5521 - val_tumor_type_loss: 0.4845 - learning_rate: 5.0000e-04
Epoch 9/50
143/143 ━━━━━━━━━━━━━━━━━━━━ 25s 172ms/step - loss: 0.4856 - tumor_presence_accuracy: 0.9557 - tumor_presence_auc: 0.9886 - tumor_presence_loss: 0.0322 - tumor_presence_precision: 0.9646 - tumor_presence_recall: 0.9745 - tumor_type_accuracy: 0.6216 - tumor_type_loss: 0.3487 - val_loss: 0.7321 - val_tumor_presence_accuracy: 0.9493 - val_tumor_presence_auc: 0.9869 - val_tumor_presence_loss: 0.0378 - val_tumor_presence_pre

143/143 ━━━━━━━━━━━━━━━━━━━━ 40s 276ms/step - loss: 0.4407 - tumor_presence_accuracy: 0.9561 - tumor_presence_auc: 0.9876 - tumor_presence_loss: 0.0331 - tumor_presence_precision: 0.9693 - tumor_presence_recall: 0.9701 - tumor_type_accuracy: 0.6267 - tumor_type_loss: 0.3135 - val_loss: 0.6559 - val_tumor_presence_accuracy: 0.6999 - val_tumor_presence_auc: 0.9719 - val_tumor_presence_loss: 0.1486 - val_tumor_presence_precision: 0.9839 - val_tumor_presence_recall: 0.5934 - val_tumor_type_accuracy: 0.6229 - val_tumor_type_loss: 0.3780 - learning_rate: 5.0000e-04
Epoch 12/50
143/143 ━━━━━━━━━━━━━━━━━━━━ 25s 173ms/step - loss: 0.4600 - tumor_presence_accuracy: 0.9558 - tumor_presence_auc: 0.9880 - tumor_presence_loss: 0.0331 - tumor_presence_precision: 0.9693 - tumor_presence_recall: 0.9697 - tumor_type_accuracy: 0.6275 - tumor_type_loss: 0.3284 - val_loss: 2.8491 - val_tumor_presence_accuracy: 0.9379 - val_tumor_presence_auc: 0.9709 - val_tumor_presence_loss: 0.0530 - val_tumor_presence_pr

143/143 ━━━━━━━━━━━━━━━━━━━━ 130s 911ms/step - loss: 0.3959 - tumor_presence_accuracy: 0.9619 - tumor_presence_auc: 0.9895 - tumor_presence_loss: 0.0301 - tumor_presence_precision: 0.9753 - tumor_presence_recall: 0.9721 - tumor_type_accuracy: 0.6431 - tumor_type_loss: 0.2814 - val_loss: 0.4374 - val_tumor_presence_accuracy: 0.9659 - val_tumor_presence_auc: 0.9920 - val_tumor_presence_loss: 0.0322 - val_tumor_presence_precision: 0.9864 - val_tumor_presence_recall: 0.9660 - val_tumor_type_accuracy: 0.6343 - val_tumor_type_loss: 0.3040 - learning_rate: 2.5000e-04
Epoch 19/50
143/143 ━━━━━━━━━━━━━━━━━━━━ 25s 170ms/step - loss: 0.3668 - tumor_presence_accuracy: 0.9657 - tumor_presence_auc: 0.9923 - tumor_presence_loss: 0.0264 - tumor_presence_precision: 0.9765 - tumor_presence_recall: 0.9765 - tumor_type_accuracy: 0.6460 - tumor_type_loss: 0.2619 - val_loss: 0.8394 - val_tumor_presence_accuracy: 0.9353 - val_tumor_presence_auc: 0.9900 - val_tumor_presence_loss: 0.0406 - val_tumor_presence_p

2026/02/03 15:37:20 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/03 15:38:09 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /tmp/tmpkb3u3wko/model, flavor: tensorflow). Fall back to return ['tensorflow==2.19.0', 'cloudpickle==3.1.1']. Set logging level to DEBUG to see the full traceback. 
Registered model 'BrainTumorMRI_DenseNet121_2Head' already exists. Creating a new version of this model...
2026/02/03 15:38:54 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: BrainTumorMRI_DenseNet121_2Head, version 16
Created version '16' of model 'BrainTumorMRI_DenseNet121_2Head'.
/tmp/ipykernel_55/1507103595.py:39: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecat

🏃 View run DenseNet121freeze=True_mask=True_20260203-1523 at: https://preachingly-nonabjuratory-marget.ngrok-free.dev/#/experiments/830660600119881173/runs/08b2779586e845c5a145e6fd927966b0
🧪 View experiment at: https://preachingly-nonabjuratory-marget.ngrok-free.dev/#/experiments/830660600119881173


## Epoch filter

In [29]:
history_df = pd.DataFrame(history.history)
history_df["epoch"] = history_df.index
#history_df.head(5)

,loss,tumor_presence_accuracy,tumor_presence_auc,tumor_presence_loss,tumor_presence_precision,tumor_presence_recall,tumor_type_accuracy,tumor_type_loss,val_loss,val_tumor_presence_accuracy,val_tumor_presence_auc,val_tumor_presence_loss,val_tumor_presence_precision,val_tumor_presence_recall,val_tumor_type_accuracy,val_tumor_type_loss,learning_rate,epoch
0,1.069174,0.838039,0.883219,0.120715,0.889296,0.885515,0.527030,0.729281,1.634530,0.913386,0.954329,0.062012,0.926973,0.955097,0.328959,1.177740,0.001,0
1,0.752997,0.898008,0.943807,0.074951,0.919810,0.940480,0.569709,0.522273,3.253572,0.364829,0.929100,0.693089,0.980392,0.121359,0.345582,1.901832,0.001,1
2,0.664552,0.913986,0.956230,0.064638,0.931548,0.950501,0.581090,0.457725,1.154526,0.611549,0.970522,0.181006,0.989691,0.466019,0.484689,0.726377,0.001,2
3,0.608361,0.926242,0.966741,0.057468,0.939619,0.959308,0.597943,0.420820,2.343429,0.795276,0.905030,0.341025,0.779356,0.998786,0.393701,1.491010,0.001,3
4,0.601232,0.934340,0.975398,0.047107,0.946317,0.963559,0.591158,0.423010,1.674846,0.871391,0.946117,0.099159,0.848610,1.000000,0.406824,1.175723,0.001,4


In [30]:
metrics_cols = [
    "epoch",
    "val_tumor_presence_recall",
    "val_tumor_type_accuracy",
    "val_tumor_presence_loss",
    "val_tumor_type_loss"
]

df = history_df[metrics_cols].copy()

In [31]:
df = df[
    (df["val_tumor_presence_recall"] >= 0.94) &
    (df["val_tumor_type_accuracy"] >= 0.55)
]
df

,epoch,val_tumor_presence_recall,val_tumor_type_accuracy,val_tumor_presence_loss,val_tumor_type_loss
7,7,0.991505,0.552056,0.044848,0.484504
8,8,0.956311,0.562555,0.037778,0.535217
17,17,0.966019,0.634296,0.032176,0.304005
21,21,0.985437,0.632546,0.043581,0.344266


In [32]:
def normalize(col):
    return (col - col.min()) / (col.max() - col.min() + 1e-8)

df["pres_rec_norm"] = normalize(df["val_tumor_presence_recall"])
df["type_accu_norm"] = normalize(df["val_tumor_type_accuracy"])
df["pres_loss_norm"] = 1 - normalize(df["val_tumor_presence_loss"])
df["type_loss_norm"] = 1 - normalize(df["val_tumor_type_loss"])

In [39]:
df["S"] = (
    0.40 * df["pres_rec_norm"]
  + 0.35 * df["type_accu_norm"]
  + 0.15 * df["pres_loss_norm"]
  + 0.10 * df["type_loss_norm"]
)
df

,epoch,val_tumor_presence_recall,val_tumor_type_accuracy,val_tumor_presence_loss,val_tumor_type_loss,pres_rec_norm,type_accu_norm,pres_loss_norm,type_loss_norm,S
7,7,0.991505,0.552056,0.044848,0.484504,1.000000,0.000000,7.891726e-07,2.193383e-01,0.421934
8,8,0.956311,0.562555,0.037778,0.535217,0.000000,0.127659,5.579600e-01,4.325032e-08,0.128375
17,17,0.966019,0.634296,0.032176,0.304005,0.275861,1.000000,1.000000e+00,1.000000e+00,0.710344
21,21,0.985437,0.632546,0.043581,0.344266,0.827587,0.978724,9.994406e-02,8.258692e-01,0.771166


In [40]:
best_row = df.sort_values("S", ascending=False).iloc[0]
best_epoch = int(best_row["epoch"])

print(f"✅ Best epoch selected manually: {best_epoch}")
print(best_row)

✅ Best epoch selected manually: 21
epoch                        21.000000
val_tumor_presence_recall     0.985437
val_tumor_type_accuracy       0.632546
val_tumor_presence_loss       0.043581
val_tumor_type_loss           0.344266
pres_rec_norm                 0.827587
type_accu_norm                0.978724
pres_loss_norm                0.099944
type_loss_norm                0.825869
S                             0.771166
Name: 21, dtype: float64


In [49]:
mlflow.set_tag("selection_method", "composite_score_S")
mlflow.log_metric("S", best_row.iloc[-1])
mlflow.log_metric("Best epoch", best_row.iloc[0])

In [ ]:
model.load_weights("path/to/best_epoch_21_weights")

In [ ]:
mlflow.keras.log_model(model, name="DenseNet121freeze=True_mask=True_20260203-1523")

In [35]:
Warning : do not forget :
- stratégie de fine-tuning du backbone
- BatchNormalization precaution in fine-tuning

SyntaxError: invalid character '→' (U+2192) (1940605460.py, line 4)